**SAPIANS** · SCC5819 · MÓDULO 00 — A BASE SRAG · O PRATA

# Módulo 00 — tratamento dos dados de SRAG, do Bronze ao Silver

Base: microdados do SIVEP-Gripe, notificação individual de Síndrome Respiratória
Aguda Grave. **4.109.567 registros, 194 colunas, 2019–2024.**

Este caderno percorre o tratamento campo a campo e mostra a evidência de cada
decisão, em **2023** — um ano de tamanho manejável. As afirmações sobre a série
inteira vivem no [internals](srag_silver_internals.ipynb), que varre os seis anos:
medir num ano e afirmar sobre os seis foi o erro que mais se repetiu aqui, e a
separação é a correção.

O tratamento vive em [`tools/srag_30_silver.py`](../../../tools/srag_30_silver.py) e é
**importado, não copiado** — os módulos seguintes precisam compartilhar um
tratamento só.

A referência é o script oficial do Ministério da Saúde
([MIT](https://gitlab.com/cgcovid/dados-publicos), CGCOVID/DEDT/SVSA, mesma
equipe do InfoGripe). Onde ele resolve, adotamos; onde divergimos, o dicionário
`DEVIATIONS` do módulo registra o porquê, com o número que justifica.

**Roteiro** — 1 procedência · 2 conformidade · 3 ausência · 4 plausibilidade ·
5 unicidade · 6 derivadas · 7 o que falta.

[README do módulo](../README.md) · [o Prata, por dentro](srag_silver_internals.ipynb) · [validação externa](srag_infogripe_validation.ipynb) · [`COLUMNS.md`](../COLUMNS.md) · [`QUALITY.md`](../QUALITY.md)

---

In [1]:
%pip install -q pandas pyarrow matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib
import subprocess
import sys
import tempfile
import urllib.request

import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.width", 110)

ANO = 2023
NOME = f"INFLUD{ANO % 100:02d}-26-06-2025.parquet"
DATA = pathlib.Path.home() / "Documents/srag-data"
if not (DATA / NOME).exists():
    DATA = pathlib.Path(tempfile.gettempdir()) / "srag-data"
    DATA.mkdir(parents=True, exist_ok=True)
BRONZE = DATA / NOME
if not BRONZE.exists():
    urllib.request.urlretrieve(
        f"https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/{ANO}/{NOME}", BRONZE)

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "README.md").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "README.md").exists():
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))

import sapians as SP
import srag_30_silver as S

meta = pq.ParquetFile(BRONZE).metadata
print(f"{NOME}: {SP.pt_int(meta.num_rows)} registros x {meta.num_columns} colunas")

INFLUD23-26-06-2025.parquet: 279.453 registros x 194 colunas


## §1 — Procedência

Os arquivos são a **reexportação de 26/06/2025** de todos os anos no layout atual
de 194 colunas — não o que foi publicado à época. Isso importa: o CSV de 2021
publicado em 2023 tem 166 colunas e codifica `FATOR_RISC` como `1`/`2`, enquanto
esta safra usa `1`/vazio. Comparação com literatura que usou outra safra precisa
declarar qual.

O tratamento declara seus desvios em relação ao script oficial. Cada um carrega
a medida que o justifica.

In [3]:
for nome, razao in S.DEVIATIONS.items():
    print(f"■ {nome}\n  {razao}\n")

■ caso_srag_nao_filtra
  MS line 260 runs filter(caso_srag == 1). Here it is a column. The filter drops 1,656,396 of 4,109,567 rows (40.3%), including 892,168 with CLASSI_FIN=5 and 524,862 with PCR_SARS2 positive — it selects on symptom fields being filled, not on disease.

■ coorte_sem_evolucao
  MS line 246 defines criterio_1 as HOSPITAL==1 | EVOLUCAO==2. Membership would then depend on the outcome: 24,475 rows qualify only because the patient died. `coorte_hospitalizado` uses HOSPITAL==1 alone; the full official flag stays as `caso_srag_ms`.

■ idade_neonato_zero
  MS lines 269-276 set idade_anos to NA when DT_SIN_PRI == DT_NASC. That is 6,835 neonates symptomatic on the day of birth — clinically ordinary. Age 0 describes them; absent does not.

■ datas_iso
  MS line 231 parses dates as %d/%m/%Y. The published data — CSV and parquet alike — is ISO yyyy-mm-dd, so as.Date() would return NA for every date and silently void idade_anos. DT_VGM and DT_RT_VGM really are dd/mm/yyyy and are 

## §2 — Conformidade de valor

### 2.1 O defeito que motivou o tratamento inteiro

A serialização muda entre anos: o mesmo campo é `'1'` num ano e `'1.0'` no
seguinte. Uma regra escrita como comparação literal funciona em cinco anos e
destrói o sexto — **sem mudar a contagem de linhas**, que é o modo mais
silencioso de destruir dado.

In [4]:
for col in ("PCR_SARS2", "AN_SARS2", "RAIOX_RES"):
    v = pq.read_table(BRONZE, columns=[col]).to_pandas()[col]
    print(f"{col:<11} {ANO}: {sorted(v.dropna().astype(str).str.strip().unique())[:4]}")

PCR_SARS2   2023: ['1']
AN_SARS2    2023: ['1']
RAIOX_RES   2023: ['1.0000000000', '2.0000000000', '3', '3.0000000000']


In [5]:
print(f"colunas com decimal real, isentas da normalização: {S.DECIMAL_COLUMNS}")
print(f"regex aplicado antes de inferir domínio: {S.TRAILING_ZEROS.pattern}")

colunas com decimal real, isentas da normalização: {'OBES_IMC'}
regex aplicado antes de inferir domínio: ^(-?\d+)\.0+$


O mesmo `PCR_SARS2` que aqui é `'1'` chega como `'1.0'` em 2020, e o
`RAIOX_RES` acima mostra por que o regex é `^(-?\d+)\.0+$` e não `\.0$`.
Escrito da forma curta, ele nunca teria tocado `1.0000000000`. O `OBES_IMC` fica
de fora da normalização: ali o decimal é real.

Quantas colunas mudam de forma, e o que a regra literal custaria em 2020, estão
medidos no [internals §2](srag_silver_internals.ipynb) — levantamento completo
das 194 em [`PROFILE.md`](../PROFILE.md), gerado por
[`tools/srag_20_profile.py`](../../../tools/srag_20_profile.py), que **não aceita
parâmetro de ano**, justamente para que nenhum achado fique escopado a um ano por
acidente.

### 2.2 Checkboxes: o vazio é "não marcado", não "faltante"

Dezessete campos de agente etiológico são caixas de seleção: preenchidas com `1`
quando marcadas e deixadas vazias quando não. Quatro campos com prefixo parecido
**não** são checkbox — são campos de valor com domínio próprio, e tratá-los como
caixa inverteria o significado.

In [6]:
print(f"checkbox ({len(S.CHECKBOXES)}): {', '.join(S.CHECKBOXES)}")
print(f"\nNÃO são checkbox: {', '.join(S.NOT_CHECKBOXES)}")
d = pq.read_table(BRONZE, columns=["PCR_FLUASU"]).to_pandas()["PCR_FLUASU"]
print(f"\nPCR_FLUASU, domínio observado em {ANO}: "
      f"{sorted(S.normalise(d, 'PCR_FLUASU').dropna().unique())}")
print("  1=H1N1  2=H3N2  4=não subtipável  5,6=inconclusivo")

checkbox (18): PCR_SARS2, PCR_VSR, PCR_ADENO, PCR_METAP, PCR_BOCA, PCR_RINO, PCR_PARA1, PCR_PARA2, PCR_PARA3, PCR_PARA4, PCR_OUTRO, AN_SARS2, AN_VSR, AN_ADENO, AN_PARA1, AN_PARA2, AN_PARA3, AN_OUTRO



NÃO são checkbox: TP_FLU_PCR, TP_FLU_AN, PCR_FLUASU, PCR_FLUBLI



PCR_FLUASU, domínio observado em 2023: ['1', '2', '3', '4', '5', '6']
  1=H1N1  2=H3N2  4=não subtipável  5,6=inconclusivo


### 2.3 Sete datas que não se chamavam datas

A regra de prefixo `DT_` encontra as datas ISO. Nove campos não a seguem:
o par de viagem é dd/mm/aaaa documentado, as **seis datas de dose de vacina
COVID não têm prefixo nenhum**, e `VG_DTRES` é data que o prefixo não vê.
Ficaram como texto no primeiro Silver — 138.595 primeiras doses só em 2023.

In [7]:
for c in [c for c in S.DMY_DATES if c.startswith("DOSE") or c.startswith("DOS_")]:
    v = pq.read_table(BRONZE, columns=[c]).to_pandas()[c]
    parsed = S.parse_dates(S.normalise(v, c), c)
    print(f"{c:<11} preenchidos {SP.pt_int(int(v.notna().sum())):>7}   "
          f"parse dd/mm/aaaa "
          f"{SP.pct(parsed.notna().sum() / max(int(v.notna().sum()), 1), 2):>7}")

DOSE_1_COV  preenchidos 138.597   parse dd/mm/aaaa 100,00%


DOSE_2_COV  preenchidos 126.312   parse dd/mm/aaaa 100,00%


DOSE_REF    preenchidos  84.261   parse dd/mm/aaaa 100,00%


DOSE_2REF   preenchidos  48.674   parse dd/mm/aaaa 100,00%


DOSE_ADIC   preenchidos   5.057   parse dd/mm/aaaa 100,00%


DOS_RE_BI   preenchidos  20.889   parse dd/mm/aaaa 100,00%


### 2.4 A semana que o Ministério quer dizer

O primeiro Silver derivava a semana epidemiológica com `isocalendar().week` —
a semana ISO, que começa na segunda. O SIVEP usa a semana **MMWR**, que começa
no domingo. A base publica o gabarito na coluna ao lado:

In [8]:
d = pq.read_table(BRONZE, columns=["DT_SIN_PRI", "SEM_PRI"]).to_pandas()
dt = S.parse_dates(S.normalise(d["DT_SIN_PRI"], "DT_SIN_PRI"), "DT_SIN_PRI")
sem = pd.to_numeric(S.normalise(d["SEM_PRI"], "SEM_PRI"), errors="coerce")
n = len(d)
iso = dt.dt.isocalendar().week
mmwr = S.epiweek(dt)
print(f"SEM_PRI == semana ISO  : {SP.pt_int(int((sem == iso).sum())):>7}/{SP.pt_int(n)}"
      f" = {SP.pct(int((sem == iso).sum()) / n, 2):>7}")
print(f"SEM_PRI == semana MMWR : {SP.pt_int(int((sem == mmwr).sum())):>7}/{SP.pt_int(n)}"
      f" = {SP.pct(int((sem == mmwr).sum()) / n, 2):>7}")

SEM_PRI == semana ISO  : 239.256/279.453 =  85,62%
SEM_PRI == semana MMWR : 279.453/279.453 = 100,00%


Uma derivação de uma linha, com cara de óbvia, errada para um registro em
cada sete — em todos os seis anos (internals §9). A correção é somar um dia
antes do `isocalendar()`: domingo vira segunda e a regra dos quatro dias de
janeiro vira a do MMWR. **A base sabe mais do que a intuição** — e é a coluna
que alimenta a comparação com o InfoGripe.

### 2.5 Valores fora do domínio declarado

In [9]:
dominios = {"EVOLUCAO": ["1", "2", "3", "9"], "CS_SEXO": ["M", "F", "I"],
            "UTI": ["1", "2", "9"], "CLASSI_FIN": ["1", "2", "3", "4", "5"],
            "CRITERIO": ["1", "2", "3", "4"], "HOSPITAL": ["1", "2", "9"]}
d = pq.read_table(BRONZE, columns=list(dominios)).to_pandas()
linhas = []
for c, dom in dominios.items():
    v = S.normalise(d[c], c)
    mau = v.notna() & ~v.isin(dom)
    linhas.append({"campo": c, "preenchido %": round(100 * v.notna().mean(), 1),
                   "fora do domínio": int(mau.sum()),
                   "exemplos": sorted(v[mau].unique())[:2]})
print(SP.tabela(pd.DataFrame(linhas), {"preenchido %": 1}))

     campo preenchido % fora do domínio                                 exemplos
  EVOLUCAO         95,0               0                                       []
   CS_SEXO        100,0               0                                       []
       UTI         88,2               1 [HOSPITAL NOSSA SENHORA DA CONCEICAO SA]
CLASSI_FIN         97,0               0                                       []
  CRITERIO         94,4               1                                  [";"3"]
  HOSPITAL         97,3               0                                       []


As poucas violações são datas ou texto onde deveria haver código, e todas vêm
das **mesmas linhas deslocadas em um campo** — ver §5.

## §3 — Ausência: três estados, nunca fundidos

Célula vazia não quer dizer uma coisa só. O tratamento separa três leituras e
nunca as funde:

| Estado | O que é |
|---|---|
| `nao_aplicavel` | o campo foi desabilitado por um predicado — não é dado faltante |
| `ausente` | aplicável, e deixado em branco |
| `ignorado` | `9`, registro explícito de desconhecimento |

Fundir os três é o erro mais caro da base, porque `nao_aplicavel` é a maioria.

### 3.1 A variável-funil

O sistema desabilita as treze comorbidades quando `FATOR_RISC` não declara fator
de risco ([Ribas et al., 2022](https://doi.org/10.1590/S1679-49742022000200004)).

In [10]:
d = pq.read_table(BRONZE, columns=["FATOR_RISC"] + S.COMORBIDITIES).to_pandas()
g = S.normalise(d["FATOR_RISC"], "FATOR_RISC").isin(["1", "S"])
algum = pd.concat([S.normalise(d[c], c).notna() for c in S.COMORBIDITIES], axis=1).any(axis=1)
n = len(d)
print(f"declara fator de risco      : {SP.pt_int(g.sum()):>8}"
      f" ({SP.pct(g.sum() / n, 1)} dos registros)")
print(f"não declara                 : {SP.pt_int((~g).sum()):>8}")
print(f"  destes, alguma comorbidade marcada: {SP.pt_int(algum[~g].sum())} "
      f"({SP.pct(algum[~g].sum() / (~g).sum(), 2)} dos que não declaram)")
print(f"declara → alguma marcada    : {SP.pct(algum[g].sum() / g.sum(), 1)} dos que declaram")

declara fator de risco      :  125.756 (45,0% dos registros)
não declara                 :  153.697
  destes, alguma comorbidade marcada: 1 (0,00% dos que não declaram)
declara → alguma marcada    : 98,5% dos que declaram


Zero, e o internals mostra que é zero nos seis anos. O vazio de comorbidade é
majoritariamente **campo desligado**, não informação perdida.

### 3.2 O efeito no número que se costuma reportar

In [11]:
tabela = []
for c in S.COMORBIDITIES:
    est = S.missing_state(S.normalise(d[c], c), g)
    tabela.append({
        "campo": c,
        "vazio / registros %": round(100 * est.isin(["nao_aplicavel", "ausente"]).sum() / len(d), 1),
        "não aplicável / registros %": round(100 * (est == "nao_aplicavel").sum() / len(d), 1),
        "ausência REAL / aplicáveis %": round(100 * (est == "ausente").sum() / g.sum(), 1),
    })
t = pd.DataFrame(tabela).sort_values("vazio / registros %", ascending=False)
print(SP.tabela(t, 1))
print(f"\nrazão média entre o vazio ingênuo e a ausência real: "
      f"{SP.pt((t['vazio / registros %'] / t['ausência REAL / aplicáveis %']).mean(), 1)}x")
print("os denominadores diferem de propósito: o vazio é sobre todos os registros,")
print("a ausência real só sobre aqueles em que o campo estava habilitado.")

     campo vazio / registros % não aplicável / registros % ausência REAL / aplicáveis %
  PUERPERA                71,3                        55,0                         36,2
  HEPATICA                71,0                        55,0                         35,5
 OBESIDADE                71,0                        55,0                         35,6
 SIND_DOWN                70,9                        55,0                         35,4
HEMATOLOGI                70,8                        55,0                         35,1
     RENAL                70,5                        55,0                         34,4
IMUNODEPRE                70,1                        55,0                         33,5
PNEUMOPATI                69,7                        55,0                         32,6
NEUROLOGIC                69,6                        55,0                         32,3
      ASMA                69,1                        55,0                         31,3
  DIABETES                67,8  

### 3.3 A escada de portões

O portão do `FATOR_RISC` virou regra geral: um predicado de habilitação só é
adotado se, em **cada** um dos seis anos, o filho contradiz o pai em no máximo
0,05% das células preenchidas (tier A = 0,00% cravado). O `build()` re-mede
cada portão adotado em cada ano que processa e **se recusa a rodar** se o
limite gravado quebrar — um portão medido uma vez e nunca re-medido é o erro
da asserção de um ano, de novo.

In [12]:
cols = sorted({c for c in S.GATES} | {p for p, _, _, _ in S.GATES.values()})
d = pq.read_table(BRONZE, columns=cols).to_pandas()
d = pd.DataFrame({c: S.normalise(d[c], c) for c in cols})
print(S.report(d))

gate                                    tier   recorded   measured
PUERPERA             <= FATOR_RISC   in 1/S    A     0.00%     0.001%
CARDIOPATI           <= FATOR_RISC   in 1/S    A     0.00%     0.000%
HEMATOLOGI           <= FATOR_RISC   in 1/S    A     0.00%     0.000%
SIND_DOWN            <= FATOR_RISC   in 1/S    A     0.00%     0.000%
HEPATICA             <= FATOR_RISC   in 1/S    A     0.00%     0.000%
ASMA                 <= FATOR_RISC   in 1/S    A     0.00%     0.000%
DIABETES             <= FATOR_RISC   in 1/S    A     0.00%     0.000%
NEUROLOGIC           <= FATOR_RISC   in 1/S    A     0.00%     0.000%
PNEUMOPATI           <= FATOR_RISC   in 1/S    A     0.00%     0.000%
IMUNODEPRE           <= FATOR_RISC   in 1/S    A     0.00%     0.000%
RENAL                <= FATOR_RISC   in 1/S    A     0.00%     0.000%
OBESIDADE            <= FATOR_RISC   in 1/S    A     0.00%     0.000%
OUT_MORBI            <= FATOR_RISC   in 1/S    A     0.00%     0.000%
MORB_DESC            <=

Oito predicados documentados no dicionário **falharam** a régua e estão
gravados com o pior ano em `GATES_REJECTED` — dois deles contraditos 100% das
vezes: o dicionário habilita `OUT_TRAT` quando `TIPO_TRAT = 4`, e o valor 4
**nunca ocorre** em 4,1 milhões de linhas; o bloco de viagem internacional
deixou de ser coletado depois de 2021. E o portão dos checkboxes antigênicos
é 0,00% em cinco anos e 6,40% em 2024 — o caso que uma medição de um ano
teria adotado (internals §4b).

## §4 — Plausibilidade

### 4.1 Idade — a unidade é uma coluna separada

O dicionário diz que o sistema deriva `NU_IDADE_N` das datas, e que `TP_IDADE`
indica se o número está em **dias, meses ou anos**. Ler `NU_IDADE_N` sozinho
transforma um bebê de sete dias em alguém de sete anos.

In [13]:
d = pq.read_table(BRONZE, columns=["TP_IDADE", "NU_IDADE_N", "DT_NASC", "DT_SIN_PRI"]).to_pandas()
tp = S.normalise(d["TP_IDADE"], "TP_IDADE")
rot = {"1": "1 = dias", "2": "2 = meses", "3": "3 = anos"}
print(SP.tabela(tp.map(rot).value_counts(dropna=False).to_frame("registros")
                  .assign(pct=lambda x: (100 * x.registros / len(d)).round(2)),
                {"pct": 2}, index=True))
nasc = S.parse_dates(S.normalise(d["DT_NASC"], "DT_NASC"), "DT_NASC")
sin = S.parse_dates(S.normalise(d["DT_SIN_PRI"], "DT_SIN_PRI"), "DT_SIN_PRI")
idade = (sin - nasc).dt.days / 365.25
print(f"\nidade calculada das datas — mediana {SP.pt(idade.median(), 1)}, "
      f"negativas {int((idade < 0).sum())}, acima de 120 {int((idade > 120).sum())}")

          registros    pct
TP_IDADE                  
3 = anos    213.240  76,31
2 = meses    62.834  22,48
1 = dias      3.379   1,21



idade calculada das datas — mediana 8,7, negativas 1, acima de 120 3


Seguimos o Ministério e calculamos das datas, o que contorna a unidade.
Divergimos numa regra: o script oficial **anula** a idade quando
`DT_SIN_PRI == DT_NASC`; são recém-nascidos sintomáticos no dia do parto, e
idade 0 os descreve melhor do que ausente.

As idades impossíveis ficam **registradas, não corrigidas** — não se sabe qual
das duas datas está errada.

### 4.2 Ordem entre datas

In [14]:
dcols = ["DT_SIN_PRI", "DT_NOTIFIC", "DT_INTERNA", "DT_EVOLUCA", "DT_ENCERRA"]
d = pq.read_table(BRONZE, columns=dcols).to_pandas()
p = {c: S.parse_dates(S.normalise(d[c], c), c) for c in dcols}
regras = [("DT_SIN_PRI ≤ DT_NOTIFIC", "DT_SIN_PRI", "DT_NOTIFIC"),
          ("DT_NOTIFIC ≤ DT_ENCERRA", "DT_NOTIFIC", "DT_ENCERRA"),
          ("DT_INTERNA ≤ DT_EVOLUCA", "DT_INTERNA", "DT_EVOLUCA"),
          ("DT_EVOLUCA ≤ DT_ENCERRA", "DT_EVOLUCA", "DT_ENCERRA")]
out = []
for nome, a, b in regras:
    amb = p[a].notna() & p[b].notna()
    out.append({"regra": nome, "avaliadas": int(amb.sum()),
                "cobertura %": round(100 * amb.mean(), 1),
                "violações": int((amb & (p[a] > p[b])).sum())})
print(SP.tabela(pd.DataFrame(out), {"cobertura %": 1}))

                  regra avaliadas cobertura % violações
DT_SIN_PRI ≤ DT_NOTIFIC   279.453       100,0         0
DT_NOTIFIC ≤ DT_ENCERRA   264.051        94,5         0
DT_INTERNA ≤ DT_EVOLUCA   236.210        84,5         0
DT_EVOLUCA ≤ DT_ENCERRA   244.923        87,6     6.405


A **cobertura** é parte do achado: uma regra que avalia 12% dos registros não
atesta os outros 88%. Três regras passam limpas — o sistema valida ordem na
entrada. A quarta não: `DT_EVOLUCA > DT_ENCERRA` acontece em escala, e nenhuma
checagem anterior a testava.

## §5 — Unicidade — e por que não deduplicamos

A chave oficial de deduplicação do SIVEP usa **nome, CPF e nome da mãe**, campos
que a versão pública remove por LGPD. Uma chave substituta com
`DT_NASC + CS_SEXO + município + DT_SIN_PRI` tem falso-positivo de ~24% no geral
e **~90%** nas células de município-dia mais densas, medido por permutação — ela
falha exatamente onde a epidemia acontece.

Então marcamos, não removemos. As únicas linhas que saem do Silver são as
fisicamente corrompidas.

In [15]:
silver, quarentena = S.build(BRONZE)
print(f"Bronze     : {SP.pt_int(meta.num_rows):>9}")
print(f"Silver     : {SP.pt_int(len(silver)):>9}")
print(f"Quarentena : {SP.pt_int(len(quarentena)):>9}")
assert len(silver) + len(quarentena) == meta.num_rows
print("\ninvariante: nenhuma linha perdida entre Bronze e Silver ✓")

Bronze     :   279.453
Silver     :   279.451
Quarentena :         2

invariante: nenhuma linha perdida entre Bronze e Silver ✓


In [16]:
print(SP.tabela(quarentena[["motivo", "HOSPITAL", "DT_INTERNA", "UTI", "NM_UN_INTE"]]))

                                        motivo HOSPITAL DT_INTERNA                                    UTI                                         NM_UN_INTE
linha deslocada: coluna codificada contém data     <NA>          1 HOSPITAL NOSSA SENHORA DA CONCEICAO SA                                             431490
linha deslocada: coluna codificada contém data        1 2023-12-16                                      9 HOSPITAL E MATERNIDADE EUFRASIA MARIA DA CONCEICAO


O deslocamento fica visível: `UTI` carrega um nome de hospital e `NM_UN_INTE`
carrega o código CNES. Em linha assim **todo** campo está errado, não só os que
parecem estranhos — por isso a detecção é de linha inteira, não de coluna.

## §6 — Variáveis derivadas — o catálogo oficial completo

O catálogo do Ministério resolve a co-detecção publicando **as duas versões**:
uma que a admite e outra, com sufixo `_unico`, que a exclui. Não colapsa —
duplica, e deixa a escolha para quem usa.

In [17]:
orig = set(pq.ParquetFile(BRONZE).schema_arrow.names)
der = [c for c in silver.columns if c not in orig]
grupos = {"data parseada (_d)": "_d", "checkbox (_marcado)": "_marcado",
          "estado de ausência (_estado)": "_estado", "etiologia caso": "_caso",
          "etiologia óbito": "_obito", "sem co-detecção (_unico)": "_unico"}
usados = set()
for k, suf in grupos.items():
    cols = [c for c in der if c.endswith(suf) and c not in usados]
    usados |= set(cols)
    print(f"  {k:<28} {len(cols):>3}")
print(f"  {'base e investigação':<28} {len(set(der) - usados):>3}")
print(f"  {'TOTAL derivadas':<28} {len(der):>3}")
print(f"\n  base e investigação: {', '.join(sorted(set(der) - usados))}")

  data parseada (_d)            32
  checkbox (_marcado)           18
  estado de ausência (_estado)  47
  etiologia caso                20
  etiologia óbito               20
  sem co-detecção (_unico)      40
  base e investigação           47
  TOTAL derivadas              224

  base e investigação: FAB_ADIC_codigo, FAB_ADIC_fabricante, FAB_COVRF2_codigo, FAB_COVRF2_fabricante, FAB_COVRF_codigo, FAB_COVRF_fabricante, FAB_COV_1_codigo, FAB_COV_1_fabricante, FAB_COV_2_codigo, FAB_COV_2_fabricante, FAB_RE_BI_codigo, FAB_RE_BI_fabricante, ano_epi_notificacao, ano_epi_primeiro_sinto, ano_sintomas, caso_srag_ms, cod_idade_consistente, codeteccao_casos, coorte_hospitalizado, dias_ate_internacao, dias_uti, dose_1_covid_antes_campanha, fator_risco_declarado, idade_anos, idade_cat_ms, idade_declarada_anos, idade_unidade, investigacao, municipio_inte_df_ra, municipio_inte_valido, municipio_notif_df_ra, municipio_notif_valido, municipio_resid_df_ra, municipio_resid_valido, n_detectado, n_doses_

In [18]:
agentes = sorted({c[: -len("_caso")] for c in silver.columns if c.endswith("_caso")})
res = pd.DataFrame({
    "agente": agentes,
    "casos": [int(silver[f"{a}_caso"].sum()) for a in agentes],
    "óbitos": [int(silver[f"{a}_obito"].sum()) for a in agentes],
    "casos sem co-detecção": [int(silver[f"{a}_caso_unico"].sum()) for a in agentes],
}).sort_values("casos", ascending=False)
print(SP.tabela(res))
print(f"\nco-detecção            : {SP.pt_int(int(silver['codeteccao_casos'].sum())):>8}")
print(f"nenhum agente detectado: {SP.pt_int(int(silver['n_detectado'].sum())):>8}")
print(f"em investigação        : {SP.pt_int(int(silver['investigacao'].sum())):>8}")

                  agente  casos óbitos casos sem co-detecção
                   covid 51.003  9.492                49.349
                     vsr 28.539    480                26.063
               rinovirus 15.010    391                11.875
         influenza_geral 12.978  1.178                11.864
                     ovr 11.301    394                 7.234
       influenza_a_total  8.126    795                 7.440
       influenza_b_total  4.877    386                 4.445
influenza_b_inconclusivo  4.646    328                 4.228
       influenza_a_n_sub  4.398    288                 4.032
              metapneumo  3.395    108                 2.613
          influenza_h1n1  3.086    441                 2.843
              adenovirus  2.786     86                 1.338
            outros_virus  2.562    112                 1.051
           parainfluenza  2.464     83                 1.689
               bocavirus  1.153     33                   543
influenza_a_n_subtipavel

### 6.2 Os dois agentes que o catálogo esquecia

A primeira transcrição do catálogo parava nos testes de valor de
`PCR_FLUASU`/`PCR_FLUBLI` — e deixava de fora `influenza_a_n_sub_caso`
(`PCR_FLUASU == 3`, "não subtipado", **ou** a triagem `TP_FLU_*` positiva sem
nenhum subtipo) e `influenza_b_inconclusivo_caso`. Exatamente os dois campos
que `NOT_CHECKBOXES` nomeia para não serem maltratados — nomeados, e nunca
consumidos.

In [19]:
flu = {c: int(silver[c].sum()) for c in
       ("influenza_a_n_sub_caso", "influenza_b_inconclusivo_caso",
        "influenza_a_total_caso", "influenza_b_total_caso", "influenza_geral_caso")}
for k, v in flu.items():
    print(f"  {k:<32} {SP.pt_int(v):>7}")
print(f"\nantes da correção, influenza_geral em {ANO} contava 3.959 —")
print(f"a definição oficial conta {SP.pt_int(flu['influenza_geral_caso'])}: 3,3x,"
      f" no mesmo ano, na mesma base.")

  influenza_a_n_sub_caso             4.398
  influenza_b_inconclusivo_caso      4.646
  influenza_a_total_caso             8.126
  influenza_b_total_caso             4.877
  influenza_geral_caso              12.978

antes da correção, influenza_geral em 2023 contava 3.959 —
a definição oficial conta 12.978: 3,3x, no mesmo ano, na mesma base.


A definição de uma feature é uma decisão de modelagem tomada meses antes de
qualquer explicador rodar — e errada, ela apaga dois terços de uma classe. A
recontagem nos seis anos (33.668 → 71.808) está no internals §8, verificada
de forma independente contra a tabela do script oficial.

Uma armadilha documentada: o **dicionário publicado** dá ao `adenovirus_caso` o
critério do VSR (`AN_VSR | PCR_VSR`). O script R está correto
(`AN_ADENO | PCR_ADENO`) e é ele que implementamos — quem seguir só o PDF conta
VSR achando que conta adenovírus.

As variantes `_obito` combinam o exame com o **desfecho**. Pelo critério deste
módulo elas são matéria do Gold; ficam calculadas porque o produto oficial as
define e um módulo que reproduza o boletim precisa delas, mas **nenhum modelo
pode tomá-las como feature**.

## §7 — O que **não** foi tratado

O contrato de cobertura ([`COLUMNS.md`](../COLUMNS.md), gerado por
[`tools/srag_31_columns.py`](../../../tools/srag_31_columns.py)) afirma —
e **verifica no build** — que as 194 colunas têm regra: 13 famílias que
particionam o esquema, domínio fechado onde existe, política declarada para
texto livre e identificador, portão onde a medição confirma. A lista do que
falta encolheu para o residual real:

| Item | Situação |
|---|---|
| `DT_EVOLUCA > DT_ENCERRA` | detectado em escala, não corrigido — não se sabe qual data está errada |
| 11 datas de 2020 fora de [1900, 2030] | inclusive `1695-06-14 02:32:37.742690304` literal no parquet; visíveis, não reparadas |
| Referencial CNES | adiado por decisão (tabela grande, mensal); IBGE feito |
| Texto livre sem vocabulário pequeno | `OUTRO_DES`/`MORB_DESC` (200 mil+ distintos) ficam crus, com censo medido |
| `DT_NASC` no Silver | quase-identificador; classe `identifier`, decisão de privacidade documentada |
| Consistência resultado↔checkbox | medida e **reprovada** (RES_AN positivo sem agente: 9–20% ao ano) — registrada como checagem que falha, não escondida |

**Decisões que são do Gold, não daqui**

Definição do alvo; exclusão por vazamento — agora **gerável** da classe
`leakage` (11 colunas cruas + 36 derivadas `_obito*` + `dias_uti` +
`dias_ate_internacao` + `caso_srag_ms`); recorte de anos e coorte;
codificação dos três estados; split. O Silver não tomou nenhuma — é essa
abstenção que permite ~20 módulos compartilharem a mesma base.